In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import os

In [2]:
base_path = "../data/raw/cicids2017/"
files = os.listdir(base_path)
files

['Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
 'Friday-WorkingHours-Morning.pcap_ISCX.csv',
 'Monday-WorkingHours.pcap_ISCX.csv',
 'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',
 'Tuesday-WorkingHours.pcap_ISCX.csv',
 'Wednesday-workingHours.pcap_ISCX.csv']

In [3]:
def extract_benign(path, chunksize=50000):
    benign_chunks = []
    for chunk in tqdm(pd.read_csv(path, chunksize=chunksize, low_memory=False)):
        
        # Rename label column if needed
        if ' Label' in chunk.columns:
            chunk = chunk.rename(columns={' Label': 'Label'})
        if 'Label ' in chunk.columns:
            chunk = chunk.rename(columns={'Label ': 'Label'})
        
        # Extract only BENIGN rows
        benign = chunk[chunk['Label'] == 'BENIGN']
        
        benign_chunks.append(benign)
        
    return pd.concat(benign_chunks, ignore_index=True)

In [4]:
benign_frames = []

for f in files:
    print(f"\nExtracting from: {f}")
    benign_frames.append(extract_benign(base_path + f))


Extracting from: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv


5it [00:02,  1.98it/s]



Extracting from: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv


6it [00:02,  2.36it/s]



Extracting from: Friday-WorkingHours-Morning.pcap_ISCX.csv


...

4it [00:01,  2.65it/s]



Extracting from: Tuesday-WorkingHours.pcap_ISCX.csv


9it [00:04,  1.93it/s]



Extracting from: Wednesday-workingHours.pcap_ISCX.csv


14it [00:06,  2.11it/s]


In [5]:
benign_total = pd.concat(benign_frames, ignore_index=True)
benign_total.shape

(2273097, 79)

In [6]:
benign_total['Label'].value_counts()

Label
BENIGN    2273097
Name: count, dtype: int64

In [7]:
benign_sample = benign_total.sample(frac=0.15, random_state=42)
benign_sample.shape

(340965, 79)

In [8]:
benign_sample.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
1124308,1024,48,2,0,4,0,2,2,2.000000,0.000000,...,24,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2197469,53,260,2,2,72,454,36,36,36.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1241997,57332,52,1,1,0,0,0,0,0.000000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1266772,443,246731,11,7,587,5527,191,0,53.363636,71.315178,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
574450,443,6209,5,0,37,0,37,0,7.400000,16.546903,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [9]:
benign_sample.to_csv("../data/processed/cicids_benign_sample.csv", index=False)